# Linear Classification using Numeric Features — Spam vs Ham

Companion notebook to `3LinearClassificationNumericFeatures.md`.

**Goal:** see *why* SVMs need numeric features and *how* word-frequency features let us
draw a linear boundary between **spam (1)** and **ham (0)**.

Dataset: `Spam.csv` (spambase) — 4601 emails, 57 numeric features, 1 label.

## 1. Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PATH = 'Module Resources - SVM and Naive Bayes/SVM - Spam Classification/Spam.csv'
df = pd.read_csv(PATH)
print(df.shape)        # (4601, 58)
df.head()

## 2. Every feature is numeric

This is the SVM requirement in action — no strings, no categories, just numbers.

In [ ]:
print('Label counts (1=spam, 0=ham):')
print(df['spam'].value_counts())
print()
print('All feature dtypes numeric?',
      (df.drop(columns='spam').dtypes != object).all())
df.dtypes.value_counts()

## 3. Which words separate spam from ham?

Compare the **average word frequency** for spam vs ham emails.
Spam-leaning words light up for class 1; office words for class 0.

In [ ]:
spam = df[df.spam == 1]
ham  = df[df.spam == 0]

words = ['word_freq_free', 'word_freq_money', 'word_freq_000',
         'word_freq_credit', 'word_freq_your', 'word_freq_remove',
         'word_freq_hp', 'word_freq_george', 'word_freq_meeting',
         'word_freq_project', 'word_freq_re', 'word_freq_edu']

compare = pd.DataFrame({
    'spam_mean': spam[words].mean(),
    'ham_mean':  ham[words].mean(),
})
compare['spam_leaning'] = compare['spam_mean'] > compare['ham_mean']
compare.round(3)

In [ ]:
# Visualise the contrast
ax = compare[['spam_mean','ham_mean']].plot(kind='barh', figsize=(8,6),
                                            color=['crimson','steelblue'])
ax.set_xlabel('average word frequency (%)')
ax.set_title('Spam-leaning vs Ham-leaning words')
plt.tight_layout(); plt.show()

Notice: `free`, `money`, `000`, `credit`, `remove`, `your` are bigger in **spam**;
`hp`, `george`, `meeting`, `project`, `edu` are bigger in **ham**.
These numeric gaps are exactly what a linear boundary exploits.

## 4. Two features → a 2D picture

We can't plot 57 dimensions, so pick **two** strong features and scatter the emails.
Even in 2D the two classes pull apart.

In [ ]:
f1, f2 = 'word_freq_your', 'word_freq_000'

plt.figure(figsize=(7,6))
plt.scatter(ham[f1],  ham[f2],  s=8, alpha=.3, label='ham (0)',  color='steelblue')
plt.scatter(spam[f1], spam[f2], s=8, alpha=.3, label='spam (1)', color='crimson')
plt.xlabel(f1); plt.ylabel(f2)
plt.title('Emails as points in numeric feature space')
plt.legend(); plt.xlim(0,6); plt.ylim(0,3)
plt.show()

## 5. A linear boundary (teaser)

An SVM finds the **maximum-margin line** that best separates the points.
Here we fit a *linear* SVM on just those two features and draw its boundary.

> Full theory comes in the next note — this is just to see the line exist.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

X = df[[f1, f2]].values
y = df['spam'].values

# SVM is distance-based → standardise first (see the .md note)
Xs = StandardScaler().fit_transform(X)

clf = SVC(kernel='linear', C=1.0)
clf.fit(Xs, y)
print('Training accuracy on 2 features:', round(clf.score(Xs, y), 3))

In [ ]:
# Plot the decision boundary in standardised space
x_min, x_max = Xs[:,0].min()-.5, Xs[:,0].max()+.5
y_min, y_max = Xs[:,1].min()-.5, Xs[:,1].max()+.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7,6))
plt.contourf(xx, yy, Z, alpha=.15, cmap='coolwarm')
plt.scatter(Xs[y==0,0], Xs[y==0,1], s=8, alpha=.3, color='steelblue', label='ham')
plt.scatter(Xs[y==1,0], Xs[y==1,1], s=8, alpha=.3, color='crimson',  label='spam')
plt.xlabel(f1+' (scaled)'); plt.ylabel(f2+' (scaled)')
plt.title('Linear SVM boundary on 2 numeric features')
plt.legend(); plt.show()

## 6. Why scaling matters — a quick demonstration

`capital_run_length_total` ranges into the thousands while word frequencies are ~0–5.
Without scaling, the big-range feature dominates the distance maths.

In [ ]:
rng = df[['word_freq_free', 'capital_run_length_total']].describe().loc[['min','max']]
print('Raw feature ranges:')
print(rng)
print()
print('After standardisation both have mean~0, std~1:')
scaled = StandardScaler().fit_transform(df[['word_freq_free','capital_run_length_total']])
print(pd.DataFrame(scaled, columns=['free','cap_total']).describe().loc[['mean','std']].round(2))

## Summary

```
• SVM needs numeric features — Spam.csv already gives word/char frequencies as numbers.
• Each email = a point in feature space; spam and ham words pull the points apart.
• A linear SVM draws a maximum-margin line to separate them.
• Standardise first — SVM is distance-based, so scale matters.
```

**Next:** linear boundaries in full — the maximum-margin classifier and the role of `C`.
See [[1BiasVsVariance]] — `C` is a bias-variance knob.